Basic Components
====

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import tiktoken

### 1. Download data

In [2]:

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")


url = "https://www.gutenberg.org/ebooks/2701.txt.utf-8"
repo_root = find_repo_root()
data_dir = repo_root / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)

text_path = data_dir / "moby_dick.txt"

if not text_path.exists():
    urlretrieve(url, text_path)

text = text_path.read_text(encoding="utf-8")

print(f"Saved to: {text_path}")
print(f"Characters: {len(text):,}")
print(text[:500])


Saved to: /home/pyml/git_vmirly/llm-recipes/data/raw/moby_dick.txt
Characters: 1,238,279
The Project Gutenberg eBook of Moby Dick; Or, The Whale
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBo


### 2. Tokenization

In [3]:
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = tokenizer.encode(text)

n_vocab = tokenizer.n_vocab

# number of unique tokens in the tokenizer's vocabulary
print(f"Number of unique tokens: {n_vocab}")

print(f"Tokens: {len(token_ids):,}")
print(f"First 10 token IDs: {token_ids[:10]}")

# decoding:
decoded_text = tokenizer.decode(token_ids[:10])
print(f"Decoded text: {decoded_text}")


Number of unique tokens: 50257
Tokens: 331,691
First 10 token IDs: [464, 4935, 20336, 46566, 286, 16540, 88, 11740, 26, 1471]
Decoded text: The Project Gutenberg eBook of Moby Dick; Or


### 3. Embeddings

In [4]:
embedding_dim = 768
context_length = 1024

In [5]:
import torch
import torch.nn as nn

token_embeddings = nn.Embedding(
    num_embeddings=n_vocab, embedding_dim=embedding_dim
)

print(f"Token embeddings shape: {token_embeddings.weight.shape}")

# testing:
input_ids = torch.tensor(token_ids[:10])
tok_emb = token_embeddings(input_ids)

print(f"input_ids.shape: {input_ids.shape}")   # [10]
print(f"tok_emb.shape: {tok_emb.shape}")     # [10, 768]

tok_emb

Token embeddings shape: torch.Size([50257, 768])
input_ids.shape: torch.Size([10])
tok_emb.shape: torch.Size([10, 768])


tensor([[ 2.8755, -0.2256,  1.4472,  ..., -0.4043,  0.0142, -1.5455],
        [ 1.2819,  0.0982, -0.7333,  ...,  0.2251, -1.2938, -2.4324],
        [-0.9169,  0.9491,  1.0793,  ..., -1.5542, -1.7350,  1.8333],
        ...,
        [ 0.2036,  0.9539,  1.1105,  ...,  0.8266,  1.5051,  2.2507],
        [ 0.6189, -0.5631,  0.0594,  ...,  0.5698,  0.4052, -0.5036],
        [ 0.8548, -0.7647, -0.5330,  ...,  1.5918, -0.4567,  1.8280]],
       grad_fn=<EmbeddingBackward0>)

In [6]:
pos_embeddings = nn.Embedding(
    num_embeddings=context_length, embedding_dim=embedding_dim
)

seq_len = input_ids.shape[0]

positions = torch.arange(seq_len)
pos_emb = pos_embeddings(positions)
print(f"pos_emb.shape: {pos_emb.shape}")     # [10, 768]

pos_emb

pos_emb.shape: torch.Size([10, 768])


tensor([[ 1.3876, -0.3715,  0.5859,  ...,  0.8728,  1.6413, -1.2270],
        [-2.5959, -1.7766, -1.1258,  ..., -0.1317, -0.2138,  0.7445],
        [ 0.9451, -0.7019,  1.5919,  ...,  0.4123,  0.5633,  1.1474],
        ...,
        [-0.6394,  0.8014,  0.0843,  ..., -1.5973,  0.3010,  0.3515],
        [-2.0828, -1.6597, -0.3795,  ...,  2.4366,  0.5806, -0.0411],
        [ 0.7042,  1.5595, -0.6971,  ..., -1.9747, -0.9300,  1.6874]],
       grad_fn=<EmbeddingBackward0>)

In [7]:
# adding token embeddings and positional embeddings

x = tok_emb + pos_emb
print(f"x.shape: {x.shape}")  # [10, 768]

x.shape: torch.Size([10, 768])


### 4. Self-attention

Notes:
  - This is bidirectional attention, because every token can attend to every other token.
  - (GPT needs causal self-attention, where token i can only attend to tokens <= i)

In [14]:
# bidirectional attention

class SelfAttention(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.w_q = nn.Linear(embedding_dim, embedding_dim)
        self.w_k = nn.Linear(embedding_dim, embedding_dim)
        self.w_v = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (q.shape[-1] ** 0.5)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_weights, v)

        return output, attn_weights


# testing the self-attention layer
attention_layer = SelfAttention(embedding_dim=embedding_dim)
output, attn_weights = attention_layer(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

print(f"Attention weights shape: {attn_weights.shape}")
print(
    "Sum of attention weights for each token (should be 1):\n",
    attn_weights.sum(dim=-1)
)

Input shape: torch.Size([10, 768])
Output shape: torch.Size([10, 768])
Attention weights shape: torch.Size([10, 10])
Sum of attention weights for each token (should be 1):
 tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000], grad_fn=<SumBackward1>)


In [21]:
# causal self-attention

class CausalSelfAttention(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.w_q = nn.Linear(embedding_dim, embedding_dim)
        self.w_k = nn.Linear(embedding_dim, embedding_dim)
        self.w_v = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        attn_mask = torch.triu(
            torch.ones(x.shape[0], x.shape[0]), diagonal=1
        ).to(dtype=torch.bool, device=x.device)

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (q.shape[-1] ** 0.5)
        # apply the mask
        attn_scores = attn_scores.masked_fill(attn_mask, float("-inf"))

        attn_weights = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_weights, v)

        return output, attn_weights
    
# testing
causal_attention_layer = CausalSelfAttention(embedding_dim=embedding_dim)
output_causal, attn_weights = causal_attention_layer(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output_causal.shape}")

print(
    "Sum of attention weights for each token (should be 1):\n",
    attn_weights.sum(dim=-1)
)

# verify that the attention weights are zero for future tokens
print(
    "Attention weights must be zero for future tokens:\n",
    attn_weights
)

Input shape: torch.Size([10, 768])
Output shape: torch.Size([10, 768])
Sum of attention weights for each token (should be 1):
 tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000], grad_fn=<SumBackward1>)
Attention weights must be zero for future tokens:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.1182, 0.8818, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.5698, 0.1698, 0.2604, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.3805, 0.1769, 0.1518, 0.2908, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.2290, 0.3159, 0.1002, 0.1599, 0.1950, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0808, 0.1910, 0.2633, 0.2234, 0.1482, 0.0934, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0839, 0.2108, 0.0600, 0.1445, 0.3143, 0.1066, 0.0798, 0.0000, 0.0000,
         0.0000],
        

## References

 * OpenAI GPT-2 model (TensorFlow): https://github.com/openai/gpt-2/blob/master/src/model.py